# Exploratory Data Analysis � Home Credit Default Risk

**Goal:** Understand the dataset used in the self-healing ML pipeline.

| File | Purpose |
|------|---------|
| `data/historical_data.csv` | 80% split. Model training & baseline drift profile |
| `data/current_data.csv` | 20% split. Simulated production data (used for drift detection) |

**Target column:** `TARGET` � 0 = Repay, 1 = Default

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

DATA_DIR = os.path.join("..", "data")
hist_df = pd.read_csv(os.path.join(DATA_DIR, "historical_data.csv"))
curr_df = pd.read_csv(os.path.join(DATA_DIR, "current_data.csv"))

print("Historical shape :", hist_df.shape)
print("Current shape    :", curr_df.shape)

: 

## 1. Schema & Basic Statistics

In [ ]:
hist_df.info()

In [ ]:
hist_df.describe().T.style.background_gradient(cmap="Blues", subset=["mean", "std"])

## 2. Missing Value Check

In [ ]:
print("Missing values in historical_data.csv:")
print(hist_df.isnull().sum()[hist_df.isnull().sum() > 0])
print("\nMissing values in current_data.csv:")
print(curr_df.isnull().sum()[curr_df.isnull().sum() > 0])

## 3. Target Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, df, title in zip(axes, [hist_df, curr_df], ["Historical", "Current"]):
    counts = df["TARGET"].value_counts().sort_index()
    ax.bar(["Repay (0)", "Default (1)"], counts, color=["steelblue", "coral"])
    ax.set_title(f"{title} � TARGET Distribution")
    ax.set_ylabel("Count")
    for i, v in enumerate(counts):
        ax.text(i, v + 2, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

print("Historical class balance (%):")
print((hist_df["TARGET"].value_counts(normalize=True) * 100).round(1))

## 4. Feature Distributions (Top Numeric)

In [ ]:
num_features = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "DAYS_BIRTH", "DAYS_EMPLOYED", "EXT_SOURCE_2"]
existing_features = [f for f in num_features if f in hist_df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(existing_features):
    ax = axes[i]
    # Use 99th percentile to clip extreme outliers for visualization
    cap = hist_df[col].quantile(0.99)
    plot_df = hist_df[hist_df[col] < cap]
    
    plot_df[plot_df["TARGET"] == 0][col].plot.hist(
        ax=ax, bins=25, alpha=0.6, color="steelblue", label="Repay", density=True
    )
    plot_df[plot_df["TARGET"] == 1][col].plot.hist(
        ax=ax, bins=25, alpha=0.6, color="coral", label="Default", density=True
    )
    ax.set_title(col)
    ax.set_xlabel("")
    ax.legend(fontsize=7)

plt.suptitle("Feature Distributions by Class (Clipped at 99th Pct) � Historical Data", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
numeric_df = hist_df.select_dtypes(include=['number'])
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Pearson Correlation � Historical Data", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Distribution Drift � Historical vs. Current

The `current_data.csv` has a subtle distribution shift on **AMT_INCOME_TOTAL** and **AMT_CREDIT** applied during data preparation to simulate real-world drift (e.g. inflation, policy changes).

In [ ]:
drift_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT"]
drift_cols = [c for c in drift_cols if c in hist_df.columns]

fig, axes = plt.subplots(1, len(drift_cols), figsize=(12, 4))
if len(drift_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, drift_cols):
    cap = hist_df[col].quantile(0.95)
    hist_plot = hist_df[hist_df[col] < cap]
    curr_plot = curr_df[curr_df[col] < cap]
    
    hist_plot[col].plot.hist(
        ax=ax, bins=30, alpha=0.6, color="steelblue", label="Historical", density=True
    )
    curr_plot[col].plot.hist(
        ax=ax, bins=30, alpha=0.6, color="coral", label="Current", density=True
    )
    ax.set_title(f"{col} � Historical vs Current")
    ax.set_ylabel("Density")
    ax.legend()

plt.suptitle("Distribution Shift (Simulated Drift)", fontsize=13)
plt.tight_layout()
plt.show()